In [13]:
import pandas as pd
import numpy as np

In [14]:
import itertools
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)

oblasts = [
    "Autonomous Republic of Crimea", "Cherkasy Oblast", "Chernihiv Oblast", "Chernivtsi Oblast",
    "Dnipropetrovsk Oblast", "Donetsk Oblast", "Ivano-Frankivsk Oblast", "Kharkiv Oblast",
    "Kherson Oblast", "Khmelnytskyi Oblast", "Kirovohrad Oblast", "Kyiv Oblast",
    "Luhansk Oblast", "Lviv Oblast", "Mykolaiv Oblast", "Odesa Oblast",
    "Poltava Oblast", "Rivne Oblast", "Sumy Oblast", "Ternopil Oblast",
    "Vinnytsia Oblast", "Volyn Oblast", "Zakarpattia Oblast", "Zaporizhzhia Oblast",
    "Zhytomyr Oblast"
]
years = range(2011, 2026)
crop_types = ['Wheat', 'Barley', 'Corn', 'Sunflower', 'Rye', 'Soybean', 'Potatoes']

oblast_populations = {
    "Donetsk Oblast": 4059000, "Dnipropetrovsk Oblast": 3096000, "Kharkiv Oblast": 2598000,
    "Lviv Oblast": 2478000, "Odesa Oblast": 2351000, "Luhansk Oblast": 2102000,
    "Autonomous Republic of Crimea": 1963000, "Kyiv Oblast": 1795000, "Zaporizhzhia Oblast": 1638000,
    "Vinnytsia Oblast": 1509000, "Poltava Oblast": 1352000, "Ivano-Frankivsk Oblast": 1351000,
    "Zakarpattia Oblast": 1244000, "Khmelnytskyi Oblast": 1228000, "Zhytomyr Oblast": 1179000,
    "Cherkasy Oblast": 1160000, "Rivne Oblast": 1141000, "Mykolaiv Oblast": 1091000,
    "Sumy Oblast": 1035000, "Ternopil Oblast": 1021000, "Volyn Oblast": 1021000,
    "Kherson Oblast": 1001000, "Chernihiv Oblast": 959000, "Kirovohrad Oblast": 903000,
    "Chernivtsi Oblast": 890000
}

# 1. Define typical yields (Tonnes per Hectare) for Ukraine crops
# Format: (Average Yield, Standard Deviation)
crop_yield_profiles = {
    'Wheat': (4.0, 0.8),
    'Barley': (3.3, 0.6),
    'Corn': (6.5, 1.5),       # Corn has highly intensive yields
    'Sunflower': (2.3, 0.4),  # Lower weight per hectare, high value
    'Rye': (2.8, 0.5),
    'Soybean': (2.4, 0.5),
    'Potatoes': (18.0, 3.0)   # Extremely high tonnage per hectare
}

# 2. Define Regional Scale Factors for Hectares (Who grows a lot vs. who grows little)
# Scale: 1.0 is baseline. Powerhouse agri-regions get higher multipliers.
region_scale = {
    "Poltava Oblast": 1.5, "Vinnytsia Oblast": 1.5, "Cherkasy Oblast": 1.4, 
    "Kirovohrad Oblast": 1.4, "Kharkiv Oblast": 1.3, "Dnipropetrovsk Oblast": 1.3,
    "Odesa Oblast": 1.2, "Zaporizhzhia Oblast": 1.1, "Kherson Oblast": 1.1,
    "Chernihiv Oblast": 1.1, "Kyiv Oblast": 1.0, "Sumy Oblast": 1.0, 
    "Khmelnytskyi Oblast": 1.2, "Ternopil Oblast": 1.1, "Zhytomyr Oblast": 0.9,
    "Rivne Oblast": 0.7, "Volyn Oblast": 0.7, "Lviv Oblast": 0.6, 
    "Ivano-Frankivsk Oblast": 0.5, "Chernivtsi Oblast": 0.4, "Zakarpattia Oblast": 0.15, # Mountainous
    "Donetsk Oblast": 0.8, "Luhansk Oblast": 0.8, "Autonomous Republic of Crimea": 0.6
}

# 3. Define Crop-to-Region Suitability Matrix (0.1 to 1.5)
# Adjusts standard area depending on how common a crop is in that specific region
suitability_matrix = {crop: {oblast: 1.0 for oblast in oblasts} for crop in crop_types}

# Sunflowers & Wheat thrive in South/East
for obj in ["Kherson Oblast", "Zaporizhzhia Oblast", "Odesa Oblast", "Mykolaiv Oblast", "Kirovohrad Oblast", "Dnipropetrovsk Oblast"]:
    suitability_matrix['Sunflower'][obj] = 1.6
    suitability_matrix['Wheat'][obj] = 1.4
    suitability_matrix['Potatoes'][obj] = 0.3  # Too dry for large scale potatoes without irrigation

# Corn & Soybeans thrive in Center/North-West
for obj in ["Poltava Oblast", "Cherkasy Oblast", "Vinnytsia Oblast", "Kyiv Oblast", "Khmelnytskyi Oblast", "Sumy Oblast", "Chernihiv Oblast"]:
    suitability_matrix['Corn'][obj] = 1.7
    suitability_matrix['Soybean'][obj] = 1.5

# Potatoes and Rye thrive in the wetter North/West
for obj in ["Zhytomyr Oblast", "Rivne Oblast", "Volyn Oblast", "Chernihiv Oblast", "Lviv Oblast"]:
    suitability_matrix['Potatoes'][obj] = 1.8
    suitability_matrix['Rye'][obj] = 1.6
    suitability_matrix['Sunflower'][obj] = 0.4  # Less sun/ideal soil here

# Base average area for an individual crop in a standard region
BASE_AREA_LOC = 45000 
BASE_AREA_SCALE = 12000

# Generate full coverage combinations
combinations = list(itertools.product(oblasts, years, crop_types))

rows = []
for oblast, year, crop in combinations:
    # Get regional and crop modifiers
    r_scale = region_scale.get(oblast, 1.0)
    c_suit = suitability_matrix[crop].get(oblast, 1.0)
    
    # Calculate unique Hectare distribution for this combination
    # Ensure it never drops below 500 hectares for realism
    area_loc = BASE_AREA_LOC * r_scale * c_suit
    area_scale = BASE_AREA_SCALE * r_scale * c_suit
    hectares = int(max(500, np.random.normal(loc=area_loc, scale=area_scale)))
    
    # Introduce slight year-on-year market fluctuation (e.g., ±15%)
    year_fluctuation = 1.0 + (np.sin(year) * 0.05) + np.random.uniform(-0.1, 0.1)
    hectares = int(hectares * year_fluctuation)
    
    # Calculate Yield (Tonnes per Hectare)
    yield_mean, yield_std = crop_yield_profiles[crop]
    # Introduce a "Year effect" (e.g., 2020 was a notable drought year in Ukraine)
    if year == 2020:
        yield_mean *= 0.82 
        
    actual_yield = max(0.5, np.random.normal(loc=yield_mean, scale=yield_std))
    
    # Calculate Production
    production_tonnes = int(hectares * actual_yield)
    
    rows.append({
        'Region': oblast,
        'Year': year,
        'CropType': crop,
        'Production_Tonnes': production_tonnes,
        'Area_Hectares': hectares
    })

df = pd.DataFrame(rows)
df['Population'] = df['Region'].map(oblast_populations)

print(df.sample(10)) # Look at a random sample to check variety

                   Region  Year   CropType  Production_Tonnes  Area_Hectares  \
1917          Sumy Oblast  2014   Potatoes            1044560          59861   
1137    Kirovohrad Oblast  2023  Sunflower             202744          90830   
2509  Zaporizhzhia Oblast  2024  Sunflower             295703         110516   
2163     Vinnytsia Oblast  2020      Wheat             162597          56651   
562        Donetsk Oblast  2016       Corn             310294          33881   
1653         Odesa Oblast  2022     Barley             222554          67584   
1949          Sumy Oblast  2019  Sunflower              72903          38746   
415     Chernivtsi Oblast  2025       Corn             186233          21523   
1984          Sumy Oblast  2024  Sunflower             131063          51385   
1905          Sumy Oblast  2013     Barley              96623          44339   

      Population  
1917     1035000  
1137      903000  
2509     1638000  
2163     1509000  
562      4059000  
1653 

In [15]:
# Ensure non-negative values for production and area
# Забезпечення невід'ємних значень для виробництва та площі
df['Production_Tonnes'] = df['Production_Tonnes'].apply(lambda x: max(0, x))
df['Area_Hectares'] = df['Area_Hectares'].apply(lambda x: max(1, x)) # Area should be at least 1

In [16]:
# Calculate Yield (Production / Area)
# Розрахунок врожайності (Виробництво / Площа)
df['Yield_Tonnes_per_Hectare'] = df['Production_Tonnes'] / df['Area_Hectares']

In [17]:
# --- Introduce intentional data issues for cleaning exercises ---
# --- Введення навмисних проблем з даними для вправ з очищення ---

n_rows = len(df)

# 1. Missing values (approx 5-10%)
# 1. Пропущені значення (приблизно 5-10%)
for col in ['Production_Tonnes', 'Area_Hectares', 'Yield_Tonnes_per_Hectare']:
    missing_indices = np.random.choice(df.index, int(n_rows * np.random.uniform(0.05, 0.1)), replace=False)
    df.loc[missing_indices, col] = np.nan

# 2. Outliers: introduce some extremely high values
# 2. Викиди: введення деяких надзвичайно високих значень
outlier_indices = np.random.choice(df.index, int(n_rows * 0.025), replace=False)
df.loc[outlier_indices, 'Production_Tonnes'] = df.loc[outlier_indices, 'Production_Tonnes'] * np.random.uniform(5, 10)

# 3. Inconsistent string data
# 3. Непослідовні текстові дані
#   - Inconsistent capitalization for CropType
#   - Непослідовна велика/мала літера для CropType
lower_case_indices = np.random.choice(df.index, int(n_rows * 0.05), replace=False)
df.loc[lower_case_indices, 'CropType'] = df.loc[lower_case_indices, 'CropType'].str.lower()

#   - Variations in Region naming (e.g., 'Kyiv region', 'Lviv Reg.')
#   - Варіації у назвах регіонів (наприклад, 'Київський регіон', 'Львівська обл.')
region_variations_indices = np.random.choice(df.index, int(n_rows * 0.05), replace=False)
df.loc[region_variations_indices, 'Region'] = df.loc[region_variations_indices, 'Region'].apply(lambda x: x.replace('Oblast', 'Reg.').lower().replace('oblast', 'region') if 'Oblast' in x else x)

# 4. Incorrect data types (e.g., some numeric columns might be objects if mixed with text in real data)
#    For this synthetic data, we ensure they are numeric, but this is a common real-world issue.
# 4. Неправильні типи даних (наприклад, деякі числові стовпці можуть бути об'єктами, якщо змішані з текстом у реальних даних).
#    Для цих синтетичних даних ми гарантуємо, що вони є числовими, але це поширена проблема в реальному світі.

print("Original Data Sample (with intentional issues):")
print("Приклад оригінальних даних (з навмисними проблемами):")

Original Data Sample (with intentional issues):
Приклад оригінальних даних (з навмисними проблемами):


In [18]:
df.to_csv('data_workshop.csv')

In [19]:
df

,Region,Year,CropType,Production_Tonnes,Area_Hectares,Population,Yield_Tonnes_per_Hectare
0,Autonomous Republic of Crimea,2011,Wheat,126646.0,32562.0,1963000,3.889380
1,Autonomous Republic of Crimea,2011,Barley,62983.0,18040.0,1963000,NaN
2,Autonomous Republic of Crimea,2011,Corn,270873.0,35403.0,1963000,7.651131
3,Autonomous Republic of Crimea,2011,Sunflower,45552.0,21796.0,1963000,2.089925
4,Autonomous Republic of Crimea,2011,rye,NaN,28884.0,1963000,1.843339
...,...,...,...,...,...,...,...
2620,Zhytomyr Oblast,2025,Corn,193229.0,36010.0,1179000,5.365982
2621,Zhytomyr Oblast,2025,Sunflower,62305.0,25836.0,1179000,2.411558
2622,Zhytomyr Oblast,2025,Rye,167937.0,49080.0,1179000,3.421699
2623,Zhytomyr Oblast,2025,Soybean,83826.0,34664.0,1179000,2.418244
